# 03. Feature Engineering & Ensemble

Public RMSE 2,564.95143을 기록한 `02_baseline_lgbm`을 개선하기 위한 실험입니다. 리더보드 점수의 단위에 맞춰 원가격 RMSE를 중심으로 최적화하며, 시간 기반 다중 검증에서 서로 다른 오차를 만드는 세 모델을 블렌딩합니다.

1. 원가격 Target + 누수 방지 지역 타깃 인코딩 LightGBM
2. 로그 Target LightGBM
3. 범주형 변수를 직접 처리하는 CatBoost
4. 세 모델의 가중 평균

지역 타깃 인코딩은 학습 행 자신의 Target을 제외한 leave-one-out 방식이며, 검증과 Test에는 과거 학습 구간의 통계만 적용합니다.

In [1]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 70)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
RANDOM_STATE = 42

In [2]:
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

DATA_DIR = PROJECT_ROOT / 'data'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'seoul_real_estate_train.csv')
test = pd.read_csv(DATA_DIR / 'seoul_real_estate_test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print('Train:', train.shape, '| Test:', test.shape)
print('Train period:', train['Transaction_YearMonth'].min(), '~', train['Transaction_YearMonth'].max())

Train: (1969, 11) | Test: (531, 10)
Train period: 202401 ~ 202512


## 1. 평가 함수와 공통 파생변수

In [3]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def rmsle(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return rmse(np.log1p(y_true), np.log1p(y_pred))

CATEGORICAL_COLS = ['Gu', 'Dong', 'Gu_Dong']

def engineer_features(df, subway_median):
    x = df.copy().reset_index(drop=True)
    x['Transaction_Year'] = x['Transaction_YearMonth'] // 100
    x['Transaction_Month'] = x['Transaction_YearMonth'] % 100
    x['Time_Index'] = (x['Transaction_Year'] - 2024) * 12 + x['Transaction_Month'] - 1
    x['Age_at_Transaction'] = x['Transaction_Year'] - x['Year_Built']
    x['Gu_Dong'] = x['Gu'].astype(str) + '_' + x['Dong'].astype(str)

    x['Subway_Distance_Missing'] = x['Distance_to_Subway'].isna().astype('int8')
    x['Distance_to_Subway'] = x['Distance_to_Subway'].fillna(subway_median)

    x['Log_Area'] = np.log(x['Exclusive_Area'])
    x['Area_Squared'] = x['Exclusive_Area'] ** 2
    x['Area_Cubed_Scaled'] = x['Exclusive_Area'] ** 3 / 10_000
    x['Floor_Squared'] = x['Floor'] ** 2
    x['Distance_Squared'] = x['Distance_to_Subway'] ** 2
    x['Age_Squared'] = x['Age_at_Transaction'] ** 2
    x['Time_Squared'] = x['Time_Index'] ** 2

    return x.drop(columns=['ID', 'Target', 'Transaction_YearMonth', 'Year_Built'], errors='ignore')

## 2. 누수 없는 전처리

일반 피처의 중앙값과 One-Hot Encoder는 학습 구간에만 fit합니다. 타깃 인코딩은 다음 규칙을 사용합니다.

- 학습 행: 자신의 Target을 제외한 leave-one-out 평균
- 검증/Test 행: 학습 구간 전체의 그룹 평균
- 표본이 적은 그룹: 전체 평균으로 smoothing
- 인코딩 대상: `Gu`, `Dong`의 거래가와 ㎡당 가격

In [4]:
def fit_unsupervised_transformer(fit_df):
    subway_median = fit_df['Distance_to_Subway'].median()
    fit_raw = engineer_features(fit_df, subway_median)
    numeric_cols = [c for c in fit_raw.columns if c not in CATEGORICAL_COLS]
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.int8)
    encoder.fit(fit_raw[CATEGORICAL_COLS])

    def transform(df):
        raw = engineer_features(df, subway_median)
        numeric = raw[numeric_cols].reset_index(drop=True)
        encoded = pd.DataFrame(
            encoder.transform(raw[CATEGORICAL_COLS]),
            columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
        )
        return pd.concat([numeric, encoded], axis=1)

    return transform, subway_median

def add_smoothed_target_encodings(fit_df, other_df, fit_raw, other_raw, prior=5.0):
    fit_reset = fit_df.reset_index(drop=True)
    other_reset = other_df.reset_index(drop=True)
    fit_encoded = fit_raw.copy()
    other_encoded = other_raw.copy()

    encoding_targets = {
        'Target_Mean': fit_reset['Target'],
        'Price_Per_Area_Mean': fit_reset['Target'] / fit_reset['Exclusive_Area'],
    }
    for suffix, values in encoding_targets.items():
        global_sum = values.sum()
        global_count = len(values)
        global_mean = global_sum / global_count
        leave_one_out_global_mean = (global_sum - values) / (global_count - 1)
        for group_col in ['Gu', 'Dong']:
            stats_source = pd.DataFrame({group_col: fit_reset[group_col], 'value': values})
            group_sum = stats_source.groupby(group_col)['value'].transform('sum')
            group_count = stats_source.groupby(group_col)['value'].transform('count')

            # 학습 행 자신의 Target을 제외한 leave-one-out 인코딩
            fit_encoded[f'{group_col}_{suffix}'] = (
                group_sum - values + prior * leave_one_out_global_mean
            ) / (group_count - 1 + prior)

            full_stats = stats_source.groupby(group_col)['value'].agg(['sum', 'count'])
            mapping = (full_stats['sum'] + prior * global_mean) / (full_stats['count'] + prior)
            other_encoded[f'{group_col}_{suffix}'] = (
                other_reset[group_col].map(mapping).fillna(global_mean).to_numpy()
            )

    return fit_encoded, other_encoded

def fit_target_encoded_transform(fit_df, other_df, prior=5.0):
    subway_median = fit_df['Distance_to_Subway'].median()
    fit_raw = engineer_features(fit_df, subway_median)
    other_raw = engineer_features(other_df, subway_median)
    fit_te, other_te = add_smoothed_target_encodings(
        fit_df, other_df, fit_raw, other_raw, prior=prior
    )
    numeric_cols = [c for c in fit_te.columns if c not in CATEGORICAL_COLS]
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.int8)
    encoder.fit(fit_te[CATEGORICAL_COLS])

    def encode(raw):
        numeric = raw[numeric_cols].reset_index(drop=True)
        categorical = pd.DataFrame(
            encoder.transform(raw[CATEGORICAL_COLS]),
            columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
        )
        return pd.concat([numeric, categorical], axis=1)

    return encode(fit_te), encode(other_te)

## 3. 모델 설정

In [5]:
COMMON_LGB_PARAMS = {
    'objective': 'regression', 'metric': 'rmse',
    'n_estimators': 4000, 'learning_rate': 0.015,
    'min_child_samples': 40, 'feature_fraction': 1.0,
    'bagging_fraction': 1.0, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'linear_tree': True, 'linear_lambda': 10.0,
    'random_state': RANDOM_STATE, 'verbosity': -1, 'n_jobs': 1,
}
ORIGINAL_LGB_PARAMS = {**COMMON_LGB_PARAMS, 'num_leaves': 3}
LOG_LGB_PARAMS = {**COMMON_LGB_PARAMS, 'num_leaves': 4}
CAT_PARAMS = {
    'iterations': 3000, 'learning_rate': 0.03, 'depth': 5,
    'l2_leaf_reg': 3.0, 'loss_function': 'RMSE', 'eval_metric': 'RMSE',
    'random_seed': RANDOM_STATE, 'random_strength': 0.5,
    'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.5,
    'verbose': False, 'allow_writing_files': False, 'thread_count': 1,
}

# 시간 Fold의 pooled OOF 예측에서 선택한 고정 가중치
BLEND_WEIGHTS = {'TargetEncoded_LGBM': 0.55, 'Log_LGBM': 0.30, 'CatBoost': 0.15}
assert np.isclose(sum(BLEND_WEIGHTS.values()), 1.0)

## 4. 시간 기반 다중 검증

In [6]:
# Test 기간을 조회하지 않고 Train의 마지막 9개 거래월만으로 Fold를 구성합니다.
train_months = np.sort(train['Transaction_YearMonth'].unique())
validation_month_groups = np.array_split(train_months[-9:], 3)
TIME_FOLDS = [
    (f'Train-derived Fold {i}', months)
    for i, months in enumerate(validation_month_groups, start=1)
]

fold_results = []
iteration_history = {'TargetEncoded_LGBM': [], 'Log_LGBM': [], 'CatBoost': []}
oof_true, oof_te, oof_log, oof_cat, oof_blend = [], [], [], [], []

for fold_name, valid_months in TIME_FOLDS:
    fit_df = train.loc[train['Transaction_YearMonth'] < valid_months.min()].copy()
    valid_df = train.loc[train['Transaction_YearMonth'].isin(valid_months)].copy()
    assert fit_df['Transaction_YearMonth'].max() < valid_df['Transaction_YearMonth'].min()

    # 1) Leave-one-out target encoding + original Target LightGBM
    X_te_fit, X_te_valid = fit_target_encoded_transform(fit_df, valid_df, prior=5.0)
    te_model = lgb.LGBMRegressor(**ORIGINAL_LGB_PARAMS)
    te_model.fit(
        X_te_fit, fit_df['Target'].reset_index(drop=True),
        eval_set=[(X_te_valid, valid_df['Target'].reset_index(drop=True))],
        callbacks=[lgb.early_stopping(150, verbose=False)],
    )
    pred_te = te_model.predict(X_te_valid)

    # 2) Target 비사용 전처리 + log Target LightGBM
    transform, subway_median = fit_unsupervised_transformer(fit_df)
    X_fit = transform(fit_df)
    X_valid = transform(valid_df)
    log_model = lgb.LGBMRegressor(**LOG_LGB_PARAMS)
    log_model.fit(
        X_fit, np.log1p(fit_df['Target']).reset_index(drop=True),
        eval_set=[(X_valid, np.log1p(valid_df['Target']).reset_index(drop=True))],
        callbacks=[lgb.early_stopping(150, verbose=False)],
    )
    pred_log = np.expm1(log_model.predict(X_valid))

    # 3) CatBoost는 학습 구간에서 만든 원본 파생변수와 범주를 직접 사용
    cat_fit = engineer_features(fit_df, subway_median)
    cat_valid = engineer_features(valid_df, subway_median)
    cat_model = CatBoostRegressor(**CAT_PARAMS)
    cat_model.fit(
        cat_fit, fit_df['Target'].reset_index(drop=True),
        cat_features=CATEGORICAL_COLS,
        eval_set=(cat_valid, valid_df['Target'].reset_index(drop=True)),
        early_stopping_rounds=150, verbose=False,
    )
    pred_cat = cat_model.predict(cat_valid)

    pred_blend = (
        BLEND_WEIGHTS['TargetEncoded_LGBM'] * pred_te
        + BLEND_WEIGHTS['Log_LGBM'] * pred_log
        + BLEND_WEIGHTS['CatBoost'] * pred_cat
    )
    y_valid = valid_df['Target'].to_numpy()
    fold_results.append({
        'Fold': fold_name, 'Train_rows': len(fit_df), 'Valid_rows': len(valid_df),
        'TE_LGBM_RMSE': rmse(y_valid, pred_te),
        'Log_LGBM_RMSE': rmse(y_valid, pred_log),
        'CatBoost_RMSE': rmse(y_valid, pred_cat),
        'Blend_RMSE': rmse(y_valid, pred_blend),
        'Blend_RMSLE': rmsle(y_valid, pred_blend),
    })
    iteration_history['TargetEncoded_LGBM'].append(te_model.best_iteration_)
    iteration_history['Log_LGBM'].append(log_model.best_iteration_)
    iteration_history['CatBoost'].append(cat_model.get_best_iteration() + 1)
    oof_true.extend(y_valid); oof_te.extend(pred_te); oof_log.extend(pred_log)
    oof_cat.extend(pred_cat); oof_blend.extend(pred_blend)

scores = pd.DataFrame(fold_results)
display(scores)

,Fold,Train_rows,Valid_rows,TE_LGBM_RMSE,Log_LGBM_RMSE,CatBoost_RMSE,Blend_RMSE,Blend_RMSLE
0,Train-derived Fold 1,1201,278,"2,150.696041","2,121.340569","2,197.186299","2,100.895856",0.055275
1,Train-derived Fold 2,1479,244,"2,678.373316","2,647.815144","2,836.877329","2,644.800383",0.081099
2,Train-derived Fold 3,1723,246,"2,537.247199","2,718.178533","2,583.173499","2,536.907840",0.071970


In [7]:
pooled_comparison = pd.DataFrame({
    'Model': ['02 Log LightGBM', 'Target-encoded LightGBM', 'CatBoost', '03 Blend'],
    'Pooled_RMSE': [
        rmse(oof_true, oof_log), rmse(oof_true, oof_te),
        rmse(oof_true, oof_cat), rmse(oof_true, oof_blend),
    ],
    'Pooled_RMSLE': [
        rmsle(oof_true, oof_log), rmsle(oof_true, oof_te),
        rmsle(oof_true, oof_cat), rmsle(oof_true, oof_blend),
    ],
}).sort_values('Pooled_RMSE')
display(pooled_comparison)
baseline_rmse = pooled_comparison.loc[pooled_comparison['Model'] == '02 Log LightGBM', 'Pooled_RMSE'].iloc[0]
blend_rmse = pooled_comparison.loc[pooled_comparison['Model'] == '03 Blend', 'Pooled_RMSE'].iloc[0]
print(f'Pooled RMSE improvement vs 02: {(baseline_rmse - blend_rmse) / baseline_rmse:.2%}')
assert blend_rmse < baseline_rmse

,Model,Pooled_RMSE,Pooled_RMSLE
3,03 Blend,"2,425.190719",0.069676
1,Target-encoded LightGBM,"2,452.656226",0.070692
0,02 Log LightGBM,"2,494.592908",0.071445
2,CatBoost,"2,538.060970",0.071504


Pooled RMSE improvement vs 02: 2.78%


## 5. 전체 Train 재학습 및 제출

각 모델의 반복 횟수는 세 시간 Fold의 최적값 중앙값을 사용합니다. 전처리와 타깃 통계는 전체 Train에만 fit하고 Test에는 transform만 적용합니다.

In [8]:
# 1) Target-encoded original-scale LightGBM
X_te_full, X_te_test = fit_target_encoded_transform(train, test, prior=5.0)
te_rounds = int(np.median(iteration_history['TargetEncoded_LGBM']))
te_final_params = {**ORIGINAL_LGB_PARAMS, 'n_estimators': te_rounds}
te_final = lgb.LGBMRegressor(**te_final_params)
te_final.fit(X_te_full, train['Target'].reset_index(drop=True))
test_pred_te = te_final.predict(X_te_test)

# 2) Log-target LightGBM
final_transform, full_median = fit_unsupervised_transformer(train)
X_full = final_transform(train)
X_test = final_transform(test)
log_rounds = int(np.median(iteration_history['Log_LGBM']))
log_final_params = {**LOG_LGB_PARAMS, 'n_estimators': log_rounds}
log_final = lgb.LGBMRegressor(**log_final_params)
log_final.fit(X_full, np.log1p(train['Target']).reset_index(drop=True))
test_pred_log = np.expm1(log_final.predict(X_test))

# 3) CatBoost
cat_full = engineer_features(train, full_median)
cat_test = engineer_features(test, full_median)
cat_rounds = int(np.median(iteration_history['CatBoost']))
cat_final_params = {**CAT_PARAMS, 'iterations': cat_rounds}
cat_final = CatBoostRegressor(**cat_final_params)
cat_final.fit(
    cat_full, train['Target'].reset_index(drop=True),
    cat_features=CATEGORICAL_COLS, verbose=False,
)
test_pred_cat = cat_final.predict(cat_test)

test_pred = np.clip(
    BLEND_WEIGHTS['TargetEncoded_LGBM'] * test_pred_te
    + BLEND_WEIGHTS['Log_LGBM'] * test_pred_log
    + BLEND_WEIGHTS['CatBoost'] * test_pred_cat,
    0, None,
)
print('Final rounds:', {'TE_LGBM': te_rounds, 'Log_LGBM': log_rounds, 'CatBoost': cat_rounds})

Final rounds: {'TE_LGBM': 1757, 'Log_LGBM': 1475, 'CatBoost': 961}


In [9]:
prediction_frame = pd.DataFrame({'ID': test['ID'], 'Target': test_pred})
submission = sample_submission[['ID']].merge(
    prediction_frame, on='ID', how='left', validate='one_to_one'
)
assert submission['Target'].notna().all()
assert len(submission) == len(sample_submission)

output_path = SUBMISSION_DIR / '03_feature_engineering_submission.csv'
submission.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
display(submission.head())
display(submission['Target'].describe())

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/03_feature_engineering_submission.csv


,ID,Target
0,TR_2427,"36,780.817637"
1,TR_0028,"47,908.615077"
2,TR_1461,"29,209.720411"
3,TR_1977,"47,077.789920"
4,TR_2351,"47,178.074467"


count      531.000000
mean    40,018.290871
std      9,623.832797
min     14,051.896060
25%     33,038.704808
50%     39,752.509241
75%     47,112.536264
max     65,520.168565
Name: Target, dtype: float64

## 해석과 주의사항

- 시간 검증에서 `02` 로그 LightGBM보다 블렌드의 pooled RMSE가 낮은지 확인한 뒤 제출합니다.
- 타깃 인코딩은 각 행 자신의 Target을 제외하며 검증·Test에는 학습 구간 통계만 적용합니다.
- 블렌드 가중치는 동일한 세 시간 Fold에서 선택했으므로 검증 점수에 모델 선택 편향이 일부 포함될 수 있습니다.
- Public Score 개선은 보장되지 않으며, 제출 결과를 실험표에 기록하고 다음 가중치 조정의 근거로 사용합니다.